# Neural Network Hyperparameters

A small TensorFlow regression exercise for comparing hidden-layer width and activation functions on the same synthetic dataset.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.layers import Dense
from tensorflow.keras.losses import MeanSquaredError
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import SGD


In [ ]:
rng = np.random.default_rng(0)
n_samples = 300

# Synthetic health-like features: age, BMI, and heart rate.
X = rng.random((n_samples, 3)) * [80, 40, 100] + [20, 15, 60]
y = 0.4 * X[:, 0] + 1.2 * X[:, 1] + 0.2 * X[:, 2] + rng.normal(0, 5, n_samples)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)


In [ ]:
def train_regression_network(hidden_units, activation, learning_rate=0.01, epochs=100):
    model = Sequential([
        Dense(hidden_units, input_shape=(3,), activation=activation),
        Dense(1, activation='linear'),
    ])
    model.compile(optimizer=SGD(learning_rate=learning_rate), loss=MeanSquaredError())
    history = model.fit(X_train, y_train, epochs=epochs, verbose=0, validation_split=0.2)
    test_loss = model.evaluate(X_test, y_test, verbose=0)
    return model, history, test_loss

experiments = [
    {'hidden_units': 10, 'activation': 'tanh'},
    {'hidden_units': 64, 'activation': 'tanh'},
    {'hidden_units': 4, 'activation': 'tanh'},
    {'hidden_units': 4, 'activation': 'relu'},
]

results = []
for config in experiments:
    model, history, test_loss = train_regression_network(**config)
    results.append({**config, 'history': history, 'test_loss': test_loss})
    print(f"{config['hidden_units']:>2} units, {config['activation']:<4} activation -> test MSE: {test_loss:.2f}")


In [ ]:
plt.figure(figsize=(9, 5))
for result in results:
    label = f"{result['hidden_units']} units, {result['activation']}"
    plt.plot(result['history'].history['val_loss'], label=label)

plt.title('Validation Loss by Network Configuration')
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.legend()
plt.tight_layout()
plt.show()


The comparison keeps the dataset, split, optimizer, and epoch count fixed so the effect of hidden units and activation function is easier to read.
